# PrimeKG — Exploratory Data Analysis

EDA on the PrimeKG biomedical knowledge graph (genes, diseases, drugs, symptoms, and proteins), run against a Neo4j instance loaded with the dataset. Originally built as a Neo4j Browser dashboard; converted here into a runnable notebook.


In [1]:
from neo4j import GraphDatabase
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../backend/.env")

driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI", "bolt://localhost:7687"),
    auth=(os.getenv("NEO4J_USERNAME", "neo4j"), os.getenv("NEO4J_PASSWORD"))
)

def run_query(query: str) -> pd.DataFrame:
    with driver.session(database=os.getenv("NEO4J_DATABASE", "neo4j")) as session:
        result = session.run(query)
        return pd.DataFrame([r.data() for r in result])


## Total nodes

*Chart type in original dashboard: table*


In [2]:
query = """
MATCH (n) RETURN count(n) AS total_nodes;
"""
run_query(query)


,total_nodes
0,129375


## Total relations

*Chart type in original dashboard: table*


In [3]:
query = """
MATCH ()-[r]->() RETURN count(r) AS total_relationships;
"""
run_query(query)


,total_relationships
0,8100128


## Node types

*Chart type in original dashboard: pie*


In [4]:
query = """
MATCH (n:Entity) RETURN n.node_type AS type, count(*) AS count ORDER BY count DESC;
"""
run_query(query)


,type,count
0,biological_process,28642
1,gene/protein,27671
2,disease,17080
3,effect/phenotype,15311
4,anatomy,14035
5,molecular_function,11169
6,drug,7957
7,cellular_component,4176
8,pathway,2516
9,exposure,818


## Relation types

*Chart type in original dashboard: bar*


In [5]:
query = """
MATCH ()-[r]->() RETURN type(r) AS relation, count(*) AS count ORDER BY count DESC LIMIT 15;
"""
run_query(query)


,relation,count
0,anatomy_protein_present,3036406
1,drug_drug,2672628
2,protein_protein,642150
3,disease_phenotype_positive,300634
4,bioprocess_protein,289610
5,cellcomp_protein,166804
6,disease_protein,160822
7,molfunc_protein,139060
8,drug_effect,129568
9,bioprocess_bioprocess,105772


## Node source breakdown

Shows which original databases (NCBI, DrugBank, MONDO, etc.) contributed the most nodes to PrimeKG.

*Chart type in original dashboard: pie*


In [6]:
query = """
MATCH (n:Entity) RETURN n.node_source AS source, count(*) AS count ORDER BY count DESC;
"""
run_query(query)


,source,count
0,GO,43987
1,NCBI,27671
2,MONDO,15813
3,HPO,15311
4,UBERON,14035
5,DrugBank,7957
6,REACTOME,2516
7,MONDO_grouped,1267
8,CTD,818


## Cross-type relationship patterns

"Pathway patterns" in the graph — like gene→gene, drug→disease, gene→disease — showing the overall shape of how different entity types interact.

*Chart type in original dashboard: table*


In [7]:
query = """
MATCH (a:Entity)-[r]->(b:Entity)
RETURN a.node_type AS from_type, type(r) AS relation, b.node_type AS to_type, count(*) AS count
ORDER BY count DESC
LIMIT 15;
"""
run_query(query)


,from_type,relation,to_type,count
0,drug,drug_drug,drug,2672628
1,gene/protein,anatomy_protein_present,anatomy,1518203
2,anatomy,anatomy_protein_present,gene/protein,1518203
3,gene/protein,protein_protein,gene/protein,642150
4,disease,disease_phenotype_positive,effect/phenotype,150317
5,effect/phenotype,disease_phenotype_positive,disease,150317
6,biological_process,bioprocess_protein,gene/protein,144805
7,gene/protein,bioprocess_protein,biological_process,144805
8,biological_process,bioprocess_bioprocess,biological_process,105772
9,gene/protein,cellcomp_protein,cellular_component,83402


## Most significant node

Node with the most overall connections.

*Chart type in original dashboard: table*


In [8]:
query = """
MATCH (n:Entity)-[r]-()
RETURN n.node_name AS name, n.node_type AS type, count(r) AS connections
ORDER BY connections DESC
LIMIT 10;
"""
run_query(query)


,name,type,connections
0,multi-cellular organism,anatomy,34710
1,small intestine,anatomy,33710
2,testis,anatomy,33486
3,fallopian tube,anatomy,33484
4,prostate gland,anatomy,33238
5,spleen,anatomy,33200
6,intestine,anatomy,33084
7,dorsolateral prefrontal cortex,anatomy,33026
8,esophagus,anatomy,32952
9,stomach,anatomy,32946


## Most connected disease

Finds which single disease has the most connections overall — likely a well-studied, complex condition with lots of known genetic/drug/symptom links.

*Chart type in original dashboard: bar*


In [9]:
query = """
MATCH (d:Entity {node_type: "disease"})-[r]-()
RETURN d.node_name AS disease, count(r) AS connections
ORDER BY connections DESC
LIMIT 10;
"""
run_query(query)


,disease,connections
0,Mendelian disease,3048
1,hereditary breast ovarian cancer syndrome,2438
2,breast neoplasm,2346
3,hereditary breast carcinoma,2188
4,breast cancer,2168
5,squamous cell carcinoma of the corpus uteri,2158
6,undifferentiated carcinoma of the corpus uteri,2158
7,schizophrenia,1952
8,colorectal cancer,1680
9,anxiety disorder,1588


## Drug-disease direct connections

Shows specific examples of drugs directly linked to diseases, and what kind of relationship connects them (treats, causes, etc.).

*Chart type in original dashboard: table*


In [10]:
query = """
MATCH (drug:Entity {node_type: "drug"})-[r]-(disease:Entity {node_type: "disease"})
RETURN drug.node_name AS drug, type(r) AS relation, disease.node_name AS disease
LIMIT 15;
"""
run_query(query)


,drug,relation,disease
0,Salmon calcitonin,off-label use,osteogenesis imperfecta
1,Salmon calcitonin,off-label use,osteogenesis imperfecta
2,Acetylcysteine,off-label use,intestinal obstruction in the newborn due to g...
3,Cysteine,off-label use,intestinal obstruction in the newborn due to g...
4,Acetylcysteine,off-label use,intestinal obstruction in the newborn due to g...
5,Cysteine,off-label use,intestinal obstruction in the newborn due to g...
6,Methoxsalen,contraindication,xeroderma pigmentosum
7,Methoxsalen,contraindication,xeroderma pigmentosum
8,Testosterone,indication,hypogonadotropic hypogonadism with or without ...
9,Testosterone undecanoate,indication,hypogonadotropic hypogonadism with or without ...


## Gene-disease-drug triangle

Finds genes connected to both a disease and a drug — e.g. MT1A is linked to squamous cell carcinoma and hepatocellular carcinoma, and also connected to Copper — a genetic link suggesting Copper might have some biological relationship to these cancers, worth exploring further even though it's not an obvious/direct connection.

*Chart type in original dashboard: table*


In [11]:
query = """
MATCH (gene:Entity {node_type: "gene/protein"})-[r1]-(disease:Entity {node_type: "disease"})
MATCH (gene)-[r2]-(drug:Entity {node_type: "drug"})
RETURN gene.node_name AS gene, disease.node_name AS disease, drug.node_name AS drug
LIMIT 15;
"""
run_query(query)


,gene,disease,drug
0,MT1A,squamous cell carcinoma,Copper
1,MT1A,hepatocellular carcinoma,Copper
2,MT1A,urinary bladder cancer,Copper
3,MT1A,pediatric hepatocellular carcinoma,Copper
4,MT1A,autoimmune hepatitis,Copper
5,MT1A,urinary bladder small cell neuroendocrine carc...,Copper
6,MT1A,urinary bladder carcinoma,Copper
7,MT1A,urinary bladder neoplasm,Copper
8,MT1A,hepatitis,Copper
9,MT1A,liver cancer,Copper


## Symptom-based disease similarity

Two different diseases that share a lot of the same symptoms — diseases with heavily overlapping symptoms can be genuinely hard for doctors to tell apart just by looking at what a patient is experiencing; this kind of analysis helps identify which diseases might get confused with each other, or which ones might share an underlying biological cause.

*Chart type in original dashboard: table*


In [ ]:
query = """
MATCH (d1:Entity {node_type: "disease"})-[:disease_phenotype_positive]->(p:Entity)<-[:disease_phenotype_positive]-(d2:Entity {node_type: "disease"})
WHERE d1.node_name < d2.node_name
WITH d1, d2, count(p) AS shared_symptoms
WHERE shared_symptoms > 10
RETURN d1.node_name AS disease_1, d2.node_name AS disease_2, shared_symptoms
ORDER BY shared_symptoms DESC
LIMIT 10;
"""
run_query(query)


## Drug safety check

For real drugs, shows their approved uses, off-label uses, AND conditions where they should be avoided — all three sides of a drug's clinical profile in one view.

*Chart type in original dashboard: table*


In [ ]:
query = """
MATCH (drug:Entity {node_type: "drug"})-[r]-(d:Entity {node_type: "disease"})
WHERE type(r) IN ["indication", "off_label_use", "contraindication"]
RETURN drug.node_name AS drug, type(r) AS relation, d.node_name AS disease
LIMIT 200;
"""
run_query(query)


## Drug-drug interaction relation types

Checks if there's a specific drug-drug interaction relationship type — drug interactions (which combinations are dangerous together) is a genuinely important, distinct medical question.

*Chart type in original dashboard: pie*


In [ ]:
query = """
MATCH ()-[r]->() WHERE type(r) CONTAINS "drug" RETURN DISTINCT type(r) AS relation_type, count(*) AS count ORDER BY count DESC;
"""
run_query(query)


## Most significant node for drug-drug interactions

Which drugs interact dangerously when taken together.

*Chart type in original dashboard: table*


In [ ]:
query = """
MATCH (d:Entity {node_type: "drug"})-[r:drug_drug]-()
RETURN d.node_name AS drug, count(r) AS interaction_count
ORDER BY interaction_count DESC
LIMIT 10;
"""
run_query(query)


## Graph Visualizations

The following queries return subgraphs best viewed as a graph visualization (e.g. in Neo4j Browser) rather than a table. They're included here for completeness — running them in this notebook returns raw node/relationship data rather than a rendered graph.


### Specific drug interactions (example: Quinidine)


In [ ]:
query = """
MATCH (d:Entity {node_name: "Quinidine", node_type: "drug"})-[r:drug_drug]-(other:Entity)
RETURN d, r, other
LIMIT 30;
"""
run_query(query)


### Visualization of most significant node (example: multi-cellular organism)


In [ ]:
query = """
MATCH (n:Entity {node_name: "multi-cellular organism"})-[r]-(m:Entity)
RETURN n, r, m
LIMIT 30;
"""
run_query(query)


### Overall cross-type relationship view


In [ ]:
query = """
MATCH (n:Entity)-[r]->(m:Entity)
WHERE n.node_type <> m.node_type
RETURN n, r, m
LIMIT 150;
"""
run_query(query)


### Specific disease comparison (shared phenotypes)


In [ ]:
query = """
MATCH (d1:Entity {node_name: "X-linked intellectual disability"})-[r1:disease_phenotype_positive]->(p:Entity)<-[r2:disease_phenotype_positive]-(d2:Entity {node_name: "developmental and epileptic encephalopathy"})
RETURN d1, r1, p, r2, d2
LIMIT 10;
"""
run_query(query)
